# Bayesian Change-Point Regimes

Models regime changes using Bayesian inference.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
ONLINE BAYESIAN CHANGE POINT MODEL (BCPM) FOR VOLATILITY REGIME DETECTION
================================================================================

Author: Trading Strategies Module
Purpose: Detect volatility regime changes in financial markets using Bayesian
         Online Changepoint Detection (BOCD) algorithm

Key Features:
- Downloads historical data from Yahoo Finance
- Computes multiple volatility and market features
- Implements online Bayesian changepoint detection
- Avoids ALL look-ahead bias (T-1 for T decision)
- Outputs regime probabilities and visualizations

References:
- Adams & MacKay (2007): "Bayesian Online Changepoint Detection"
- Original paper: https://arxiv.org/abs/0710.3742
================================================================================
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from scipy import stats
from scipy.special import logsumexp
from sklearn.decomposition import PCA
from typing import List, Tuple, Dict, Optional
import json

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


class Config:
    """
    Configuration class containing all parameters for the BCPM system.
    Modify these parameters to customize the analysis.
    """

    TICKERS = ["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "GLD", "USO"]
    VIX_TICKER = "^VIX"
    START_DATE = "2013-12-31"
    END_DATE = "2018-12-31"
    RV_WINDOW = 21
    VOLUME_MA_WINDOW = 21
    PRIOR_HAZARD_RATE = 1 / 250
    PRIOR_MU = 0.0
    PRIOR_KAPPA = 1.0
    PRIOR_ALPHA = 1.0
    PRIOR_BETA = 0.0001
    HIGH_VOL_THRESHOLD = 0.015
    REGIME_SMOOTHING_WINDOW = 5
    MIN_HISTORY = 63
    SAVE_RESULTS = True
    OUTPUT_DIR = "bcpm_results/"
    PLOT_ASSETS = ["SPY", "EEM"]


class YahooDataDownloader:
    """
    Downloads historical price data from Yahoo Finance.
    Handles both regular tickers and index data (VIX).
    """

    def __init__(self):
        """
        Initialize the data downloader.

        Parameters:
        -----------
            Yahoo Finance key for authentication
        """

    def download_ticker(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Download historical daily data for a single ticker.

        Parameters:
        -----------
        ticker : str
            Ticker symbol (e.g., 'SPY', '^VIX')
        start_date : str
            Start date in 'YYYY-MM-DD' format
        end_date : str
            End date in 'YYYY-MM-DD' format

        Returns:
        --------
        pd.DataFrame
            DataFrame with columns: date, open, high, low, close, adjClose, volume
        """
        try:
            print(f"Downloading {ticker}...")
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) == 0:
                print(f"WARNING: No data available for {ticker}")
                return pd.DataFrame()
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df["ticker"] = ticker
            print(f"  → Downloaded {len(df)} records for {ticker}")
            return df
        except Exception as e:
            print(f"ERROR downloading {ticker}: {str(e)}")
            return pd.DataFrame()

    def download_multiple_tickers(
        self, tickers: List[str], start_date: str, end_date: str
    ) -> Dict[str, pd.DataFrame]:
        """
        Download data for multiple tickers.

        Parameters:
        -----------
        tickers : List[str]
            List of ticker symbols
        start_date : str
            Start date in 'YYYY-MM-DD' format
        end_date : str
            End date in 'YYYY-MM-DD' format

        Returns:
        --------
        Dict[str, pd.DataFrame]
            Dictionary mapping ticker to DataFrame
        """
        data_dict = {}
        for ticker in tickers:
            df = self.download_ticker(ticker, start_date, end_date)
            if not df.empty:
                data_dict[ticker] = df
        print(f"\nSuccessfully downloaded {len(data_dict)}/{len(tickers)} tickers")
        return data_dict


class FeatureEngine:
    """
    Computes financial features from raw price data.
    ALL features are properly lagged to avoid look-ahead bias.
    """

    @staticmethod
    def compute_returns(df: pd.DataFrame) -> pd.DataFrame:
        """
        Compute log returns from adjusted close prices.

        Returns at time T use Close[T] and Close[T-1], so they are known
        at the END of day T (not usable for trading at day T).

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'adjClose' column

        Returns:
        --------
        pd.DataFrame
            DataFrame with added 'ret_close' column
        """
        df = df.copy()
        df["ret_close"] = np.log(df["adjClose"] / df["adjClose"].shift(1))
        return df

    @staticmethod
    def compute_realized_volatility(df: pd.DataFrame, window: int = 21) -> pd.DataFrame:
        """
        Compute realized volatility using close-to-close returns.

        RV at time T uses returns from T-window to T-1, so it's known at
        the END of day T (not usable for trading at day T).

        Formula: RV = sqrt(sum of squared returns over window)

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'ret_close' column
        window : int
            Rolling window size (default: 21 days)

        Returns:
        --------
        pd.DataFrame
            DataFrame with added 'rv_21' column
        """
        df = df.copy()
        df["rv_21"] = df["ret_close"].rolling(window=window).std() * np.sqrt(252)
        return df

    @staticmethod
    def compute_rv_change(df: pd.DataFrame) -> pd.DataFrame:
        """
        Compute first difference of realized volatility.

        This captures the CHANGE in volatility, which can signal regime shifts.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'rv_21' column

        Returns:
        --------
        pd.DataFrame
            DataFrame with added 'd_rv' column
        """
        df = df.copy()
        df["d_rv"] = df["rv_21"].diff()
        return df

    @staticmethod
    def compute_volume_ratio(df: pd.DataFrame, window: int = 21) -> pd.DataFrame:
        """
        Compute ratio of current volume to rolling average volume.

        High volume ratios can indicate regime changes (panic selling, etc.)

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'volume' column
        window : int
            Rolling window for average volume

        Returns:
        --------
        pd.DataFrame
            DataFrame with added 'volume_ratio' column
        """
        df = df.copy()
        df["volume_ma"] = df["volume"].rolling(window=window).mean()
        df["volume_ratio"] = df["volume"] / df["volume_ma"]
        return df

    @staticmethod
    def align_dataframes(data_dict: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Align multiple asset DataFrames to common dates.

        CRITICAL: Only keeps dates where ALL assets have data (no forward filling).

        Parameters:
        -----------
        data_dict : Dict[str, pd.DataFrame]
            Dictionary mapping ticker to DataFrame

        Returns:
        --------
        pd.DataFrame
            Wide-format DataFrame with aligned dates
        """
        dfs = []
        for ticker, df in data_dict.items():
            df_subset = df[
                ["date", "adjClose", "ret_close", "rv_21", "d_rv", "volume_ratio"]
            ].copy()
            df_subset = df_subset.set_index("date")
            df_subset.columns = [f"{ticker}_{col}" for col in df_subset.columns]
            dfs.append(df_subset)
        aligned_df = pd.concat(dfs, axis=1, join="inner")
        aligned_df = aligned_df.reset_index()
        return aligned_df

    @staticmethod
    def compute_pc1(aligned_df: pd.DataFrame, tickers: List[str]) -> pd.DataFrame:
        """
        Compute first principal component of returns across all assets.

        PC1 captures the main "market mode" - useful for detecting broad
        regime shifts that affect all assets.

        Parameters:
        -----------
        aligned_df : pd.DataFrame
            Aligned DataFrame with return columns
        tickers : List[str]
            List of tickers to include in PCA

        Returns:
        --------
        pd.DataFrame
            DataFrame with added 'PC1' column
        """
        df = aligned_df.copy()
        return_cols = [f"{ticker}_ret_close" for ticker in tickers]
        returns_matrix = df[return_cols].values
        valid_mask = ~np.isnan(returns_matrix).any(axis=1)
        if valid_mask.sum() < 10:
            print("WARNING: Not enough valid data for PCA")
            df["PC1"] = np.nan
            return df
        pca = PCA(n_components=1)
        pc1_values = np.full(len(df), np.nan)
        pc1_values[valid_mask] = pca.fit_transform(returns_matrix[valid_mask]).flatten()
        df["PC1"] = pc1_values
        explained_var = (
            pca.explained_variance_ratio_[0] if len(pca.explained_variance_ratio_) > 0 else 0
        )
        print(f"PC1 explains {explained_var:.2%} of return variance")
        return df

    @staticmethod
    def add_vix(aligned_df: pd.DataFrame, vix_df: pd.DataFrame) -> pd.DataFrame:
        """
        Add VIX index to aligned DataFrame.

        Parameters:
        -----------
        aligned_df : pd.DataFrame
            Aligned DataFrame with date column
        vix_df : pd.DataFrame
            VIX DataFrame with date and adjClose columns

        Returns:
        --------
        pd.DataFrame
            DataFrame with added 'VIX' column
        """
        df = aligned_df.copy()
        vix_subset = vix_df[["date", "adjClose"]].rename(columns={"adjClose": "VIX"})
        df = df.merge(vix_subset, on="date", how="left")
        return df

    @staticmethod
    def lag_features(df: pd.DataFrame, lag: int = 1) -> pd.DataFrame:
        """
        Apply lag to ALL features to prevent look-ahead bias.

        CRITICAL FOR AVOIDING LOOK-AHEAD BIAS:
        Features at time T are computed from data up to T-1, so they are NOT
        available for decision-making at time T. We must shift them by 1 day.

        This means: to make a decision at the OPEN of day T, we use features
        computed from data up to CLOSE of day T-1.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with features
        lag : int
            Number of periods to lag (default: 1)

        Returns:
        --------
        pd.DataFrame
            DataFrame with lagged features
        """
        df = df.copy()
        feature_cols = [
            col
            for col in df.columns
            if col not in ["date"]
            and (not col.endswith("_adjClose"))
            and (not col.endswith("_open"))
            and (not col.endswith("_high"))
            and (not col.endswith("_low"))
        ]
        for col in feature_cols:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
        df = df.drop(columns=feature_cols)
        return df


class BayesianOnlineChangepoint:
    """
    Implementation of Bayesian Online Changepoint Detection (BOCD) algorithm.

    Based on Adams & MacKay (2007): "Bayesian Online Changepoint Detection"

    Key Concept:
    - Maintains a probability distribution over "run length" (time since last changepoint)
    - At each time step, updates beliefs about when the last changepoint occurred
    - Uses a conjugate prior (Normal-Gamma) for efficient online updates

    The algorithm outputs:
    - P(run_length | data): Probability distribution over time since last change
    - P(changepoint): Probability that a changepoint occurred at current time
    - Regime label: High/low volatility classification
    """

    def __init__(
        self,
        hazard_rate: float = 1 / 250,
        prior_mu: float = 0.0,
        prior_kappa: float = 1.0,
        prior_alpha: float = 1.0,
        prior_beta: float = 0.0001,
    ):
        """
        Initialize BOCD with prior parameters.

        Parameters:
        -----------
        hazard_rate : float
            Prior probability of changepoint at each time step
            (1/expected_run_length). Default: 1/250 expects ~1 change per year
        prior_mu : float
            Prior mean for returns (default: 0.0)
        prior_kappa : float
            Prior precision for mean (default: 1.0)
        prior_alpha : float
            Prior shape parameter for variance (default: 1.0)
        prior_beta : float
            Prior scale parameter for variance (default: 0.0001)
        """
        self.hazard_rate = hazard_rate
        self.prior_mu = prior_mu
        self.prior_kappa = prior_kappa
        self.prior_alpha = prior_alpha
        self.prior_beta = prior_beta
        self.run_length_log_probs = None
        self.sufficient_stats = None

    def initialize(self, max_run_length: int = 1000):
        """
        Initialize the run length distribution and sufficient statistics.

        Parameters:
        -----------
        max_run_length : int
            Maximum run length to track (for computational efficiency)
        """
        self.run_length_log_probs = np.log(np.zeros(max_run_length + 1))
        self.run_length_log_probs[0] = 0
        self.sufficient_stats = {
            "n": np.zeros(max_run_length + 1),
            "mean": np.full(max_run_length + 1, self.prior_mu),
            "sum_sq_dev": np.zeros(max_run_length + 1),
        }
        self.max_run_length = max_run_length

    def update(self, observation: float) -> Tuple[float, np.ndarray]:
        """
        Update the BOCD with a new observation.

        This is the core of the online algorithm. At each time step:
        1. Compute predictive probability of observation for each run length
        2. Update run length distribution using Bayes rule
        3. Account for possibility of changepoint (run length resets to 0)
        4. Update sufficient statistics

        Parameters:
        -----------
        observation : float
            New observation (e.g., return or volatility)

        Returns:
        --------
        Tuple[float, np.ndarray]
            - Probability of changepoint at current time
            - Full run length distribution (for diagnostics)
        """
        if self.run_length_log_probs is None:
            raise ValueError("Must call initialize() before update()")
        if np.isnan(observation):
            return (0.0, np.exp(self.run_length_log_probs))
        log_pred_probs = np.zeros(self.max_run_length + 1)
        for r in range(len(self.run_length_log_probs)):
            if self.run_length_log_probs[r] > -np.inf:
                n = self.sufficient_stats["n"][r]
                mean = self.sufficient_stats["mean"][r]
                sum_sq_dev = self.sufficient_stats["sum_sq_dev"][r]
                post_kappa = self.prior_kappa + n
                post_alpha = self.prior_alpha + n / 2
                post_beta = (
                    self.prior_beta
                    + sum_sq_dev / 2
                    + self.prior_kappa * n * (mean - self.prior_mu) ** 2 / (2 * post_kappa)
                )
                df = 2 * post_alpha
                loc = mean
                scale = np.sqrt(post_beta * (post_kappa + 1) / (post_alpha * post_kappa))
                try:
                    log_pred_probs[r] = stats.t.logpdf(observation, df=df, loc=loc, scale=scale)
                except:
                    log_pred_probs[r] = -np.inf
        log_growth_probs = self.run_length_log_probs + log_pred_probs + np.log(1 - self.hazard_rate)
        log_cp_prob = logsumexp(
            self.run_length_log_probs + log_pred_probs + np.log(self.hazard_rate)
        )
        new_run_length_log_probs = np.full(self.max_run_length + 1, -np.inf)
        new_run_length_log_probs[0] = log_cp_prob
        new_run_length_log_probs[1:] = log_growth_probs[:-1]
        new_run_length_log_probs -= logsumexp(new_run_length_log_probs)
        cp_prob = np.exp(new_run_length_log_probs[0])
        new_sufficient_stats = {
            "n": np.zeros(self.max_run_length + 1),
            "mean": np.zeros(self.max_run_length + 1),
            "sum_sq_dev": np.zeros(self.max_run_length + 1),
        }
        new_sufficient_stats["n"][0] = 0
        new_sufficient_stats["mean"][0] = self.prior_mu
        new_sufficient_stats["sum_sq_dev"][0] = 0
        for r in range(1, self.max_run_length + 1):
            if new_run_length_log_probs[r] > -np.inf:
                old_r = r - 1
                old_n = self.sufficient_stats["n"][old_r]
                old_mean = self.sufficient_stats["mean"][old_r]
                old_sum_sq_dev = self.sufficient_stats["sum_sq_dev"][old_r]
                new_n = old_n + 1
                new_mean = (old_n * old_mean + observation) / new_n
                new_sum_sq_dev = old_sum_sq_dev + old_n * (observation - old_mean) ** 2 / new_n
                new_sufficient_stats["n"][r] = new_n
                new_sufficient_stats["mean"][r] = new_mean
                new_sufficient_stats["sum_sq_dev"][r] = new_sum_sq_dev
        self.run_length_log_probs = new_run_length_log_probs
        self.sufficient_stats = new_sufficient_stats
        return (cp_prob, np.exp(self.run_length_log_probs))

    def get_expected_run_length(self) -> float:
        """
        Get the expected run length (time since last changepoint).

        Returns:
        --------
        float
            Expected run length in time steps
        """
        run_lengths = np.arange(len(self.run_length_log_probs))
        probs = np.exp(self.run_length_log_probs)
        return np.sum(run_lengths * probs)

    def get_regime_volatility(self) -> float:
        """
        Get the current regime's estimated volatility.

        Returns:
        --------
        float
            Estimated volatility (standard deviation)
        """
        most_probable_r = np.argmax(self.run_length_log_probs)
        n = self.sufficient_stats["n"][most_probable_r]
        if n == 0:
            return np.nan
        sum_sq_dev = self.sufficient_stats["sum_sq_dev"][most_probable_r]
        variance = sum_sq_dev / n if n > 0 else 0
        return np.sqrt(variance)


class RegimeDetector:
    """
    Detects and classifies volatility regimes using BOCD output.
    """

    @staticmethod
    def run_bocd_on_series(
        data: np.ndarray,
        hazard_rate: float = 1 / 250,
        prior_mu: float = 0.0,
        prior_kappa: float = 1.0,
        prior_alpha: float = 1.0,
        prior_beta: float = 0.0001,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Run BOCD algorithm on a time series.

        Parameters:
        -----------
        data : np.ndarray
            Time series of observations (e.g., returns or volatility)
        hazard_rate : float
            Prior changepoint probability
        prior_mu : float
            Prior mean
        prior_kappa : float
            Prior precision for mean
        prior_alpha : float
            Prior shape for variance
        prior_beta : float
            Prior scale for variance

        Returns:
        --------
        Tuple[np.ndarray, np.ndarray, np.ndarray]
            - Array of changepoint probabilities
            - Array of expected run lengths
            - Array of regime volatilities
        """
        bocd = BayesianOnlineChangepoint(
            hazard_rate=hazard_rate,
            prior_mu=prior_mu,
            prior_kappa=prior_kappa,
            prior_alpha=prior_alpha,
            prior_beta=prior_beta,
        )
        bocd.initialize(max_run_length=min(1000, len(data)))
        cp_probs = []
        run_lengths = []
        regime_vols = []
        for obs in data:
            cp_prob, _ = bocd.update(obs)
            exp_run_length = bocd.get_expected_run_length()
            regime_vol = bocd.get_regime_volatility()
            cp_probs.append(cp_prob)
            run_lengths.append(exp_run_length)
            regime_vols.append(regime_vol)
        return (np.array(cp_probs), np.array(run_lengths), np.array(regime_vols))

    @staticmethod
    def classify_regimes(
        regime_vols: np.ndarray, threshold: float = 0.015, smoothing_window: int = 5
    ) -> np.ndarray:
        """
        Classify regimes as high/low volatility based on estimated volatility.

        Parameters:
        -----------
        regime_vols : np.ndarray
            Array of regime volatilities
        threshold : float
            Volatility threshold for classification (daily)
        smoothing_window : int
            Window for smoothing regime transitions

        Returns:
        --------
        np.ndarray
            Array of regime labels (1 = high vol, 0 = low vol)
        """
        regime_labels = (regime_vols > threshold).astype(int)
        if smoothing_window > 1:
            regime_labels_smooth = (
                pd.Series(regime_labels)
                .rolling(window=smoothing_window, min_periods=1)
                .apply(lambda x: x.mode()[0] if len(x) > 0 else x.iloc[-1], raw=False)
                .values
            )
            return regime_labels_smooth.astype(int)
        return regime_labels


class BCPMVisualizer:
    """
    Creates visualizations for BCPM results.
    """

    @staticmethod
    def plot_changepoints_and_regimes(
        df: pd.DataFrame, ticker: str, save_path: Optional[str] = None
    ):
        """
        Create comprehensive visualization of changepoints and regimes.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with price, cp_prob, and regime columns
        ticker : str
            Ticker symbol for title
        save_path : Optional[str]
            Path to save figure (if None, displays only)
        """
        fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
        ax1 = axes[0]
        price_col = f"{ticker}_adjClose"
        if price_col in df.columns:
            ax1.plot(df["date"], df[price_col], linewidth=1.5, color="black", label="Price")
            regime_col = f"{ticker}_regime"
            if regime_col in df.columns:
                high_vol_mask = df[regime_col] == 1
                ax1.fill_between(
                    df["date"],
                    df[price_col].min(),
                    df[price_col].max(),
                    where=high_vol_mask,
                    alpha=0.3,
                    color="red",
                    label="High Volatility Regime",
                )
            ax1.set_ylabel(f"{ticker} Price ($)", fontsize=11, fontweight="bold")
            ax1.legend(loc="upper left", fontsize=9)
            ax1.grid(True, alpha=0.3)
            ax1.set_title(
                f"{ticker} - Bayesian Changepoint Detection Results",
                fontsize=14,
                fontweight="bold",
                pad=15,
            )
        ax2 = axes[1]
        cp_col = f"{ticker}_cp_prob"
        if cp_col in df.columns:
            ax2.plot(df["date"], df[cp_col], linewidth=1, color="darkblue", alpha=0.7)
            ax2.fill_between(df["date"], 0, df[cp_col], alpha=0.3, color="blue")
            ax2.axhline(
                y=0.5,
                color="red",
                linestyle="--",
                linewidth=1,
                alpha=0.7,
                label="High Probability Threshold (0.5)",
            )
            ax2.set_ylabel("Changepoint\nProbability", fontsize=11, fontweight="bold")
            ax2.set_ylim([0, 1])
            ax2.legend(loc="upper right", fontsize=9)
            ax2.grid(True, alpha=0.3)
        ax3 = axes[2]
        rv_col = f"{ticker}_rv_21_lag1"
        if rv_col in df.columns:
            ax3.plot(df["date"], df[rv_col], linewidth=1.5, color="darkgreen", label="21-Day RV")
            ax3.axhline(
                y=Config.HIGH_VOL_THRESHOLD,
                color="red",
                linestyle="--",
                linewidth=1,
                alpha=0.7,
                label=f"Threshold ({Config.HIGH_VOL_THRESHOLD})",
            )
            ax3.set_ylabel("Realized\nVolatility", fontsize=11, fontweight="bold")
            ax3.legend(loc="upper left", fontsize=9)
            ax3.grid(True, alpha=0.3)
        ax4 = axes[3]
        rl_col = f"{ticker}_run_length"
        if rl_col in df.columns:
            ax4.plot(
                df["date"], df[rl_col], linewidth=1.5, color="purple", label="Expected Run Length"
            )
            ax4.set_ylabel("Run Length\n(days)", fontsize=11, fontweight="bold")
            ax4.set_xlabel("Date", fontsize=11, fontweight="bold")
            ax4.legend(loc="upper left", fontsize=9)
            ax4.grid(True, alpha=0.3)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"  → Saved plot to {save_path}")
        plt.show()

    @staticmethod
    def plot_regime_comparison(
        df: pd.DataFrame, tickers: List[str], save_path: Optional[str] = None
    ):
        """
        Compare regimes across multiple assets.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with regime columns for multiple assets
        tickers : List[str]
            List of ticker symbols
        save_path : Optional[str]
            Path to save figure
        """
        fig, ax = plt.subplots(figsize=(16, 8))
        offset = 0
        colors = plt.cm.Set3(np.linspace(0, 1, len(tickers)))
        for i, ticker in enumerate(tickers):
            regime_col = f"{ticker}_regime"
            if regime_col in df.columns:
                regime_data = df[regime_col].values + offset
                ax.fill_between(
                    df["date"],
                    offset,
                    regime_data,
                    where=df[regime_col] == 1,
                    color=colors[i],
                    alpha=0.7,
                    label=ticker,
                )
                offset += 1.5
        ax.set_ylabel("Asset (High Vol Regime Shaded)", fontsize=12, fontweight="bold")
        ax.set_xlabel("Date", fontsize=12, fontweight="bold")
        ax.set_title(
            "Volatility Regime Comparison Across Assets", fontsize=14, fontweight="bold", pad=15
        )
        ax.legend(loc="upper left", fontsize=10, ncol=2)
        ax.grid(True, alpha=0.3, axis="x")
        ax.set_yticks([])
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"  → Saved comparison plot to {save_path}")
        plt.show()


def main():
    """
    Main execution function that orchestrates the entire BCPM pipeline.
    """
    print("=" * 80)
    print("BAYESIAN CHANGE POINT MODEL (BCPM) - VOLATILITY REGIME DETECTION")
    print("=" * 80)
    print()
    print("[STEP 1] Downloading historical data from Yahoo Finance...")
    print(f"  Tickers: {Config.TICKERS}")
    print(f"  Date Range: {Config.START_DATE} to {Config.END_DATE}")
    print()
    downloader = YahooDataDownloader()
    data_dict = downloader.download_multiple_tickers(
        Config.TICKERS, Config.START_DATE, Config.END_DATE
    )
    vix_data = downloader.download_ticker(Config.VIX_TICKER, Config.START_DATE, Config.END_DATE)
    if len(data_dict) == 0:
        print("ERROR: No data downloaded. Check API key and tickers.")
        return
    print()
    print("[STEP 2] Computing features for each asset...")
    feature_engine = FeatureEngine()
    for ticker in data_dict.keys():
        print(f"  Processing {ticker}...")
        df = data_dict[ticker]
        df = feature_engine.compute_returns(df)
        df = feature_engine.compute_realized_volatility(df, window=Config.RV_WINDOW)
        df = feature_engine.compute_rv_change(df)
        df = feature_engine.compute_volume_ratio(df, window=Config.VOLUME_MA_WINDOW)
        data_dict[ticker] = df
    if not vix_data.empty:
        vix_data = feature_engine.compute_returns(vix_data)
    print("  Aligning dataframes to common dates...")
    aligned_df = feature_engine.align_dataframes(data_dict)
    print("  Computing PC1 across assets...")
    aligned_df = feature_engine.compute_pc1(aligned_df, list(data_dict.keys()))
    if not vix_data.empty:
        print("  Adding VIX index...")
        aligned_df = feature_engine.add_vix(aligned_df, vix_data)
    print("  Applying T-1 lag to all features (CRITICAL: no look-ahead bias)...")
    aligned_df = feature_engine.lag_features(aligned_df, lag=1)
    aligned_df = aligned_df.iloc[Config.MIN_HISTORY :].reset_index(drop=True)
    print(f"  → Final dataset: {len(aligned_df)} days, {len(aligned_df.columns)} columns")
    print()
    print("[STEP 3] Running Bayesian Online Changepoint Detection...")
    print(
        f"  Hazard Rate: {Config.PRIOR_HAZARD_RATE} (expects change every ~{int(1 / Config.PRIOR_HAZARD_RATE)} days)"
    )
    print(f"  High Vol Threshold: {Config.HIGH_VOL_THRESHOLD} (daily)")
    print()
    detector = RegimeDetector()
    for ticker in Config.TICKERS:
        ret_col = f"{ticker}_ret_close_lag1"
        if ret_col not in aligned_df.columns:
            print(f"  WARNING: {ticker} not found in aligned data, skipping...")
            continue
        print(f"  Detecting changepoints for {ticker}...")
        returns = aligned_df[ret_col].values
        cp_probs, run_lengths, regime_vols = detector.run_bocd_on_series(
            returns,
            hazard_rate=Config.PRIOR_HAZARD_RATE,
            prior_mu=Config.PRIOR_MU,
            prior_kappa=Config.PRIOR_KAPPA,
            prior_alpha=Config.PRIOR_ALPHA,
            prior_beta=Config.PRIOR_BETA,
        )
        regime_labels = detector.classify_regimes(
            regime_vols,
            threshold=Config.HIGH_VOL_THRESHOLD,
            smoothing_window=Config.REGIME_SMOOTHING_WINDOW,
        )
        aligned_df[f"{ticker}_cp_prob"] = cp_probs
        aligned_df[f"{ticker}_run_length"] = run_lengths
        aligned_df[f"{ticker}_regime_vol"] = regime_vols
        aligned_df[f"{ticker}_regime"] = regime_labels
        n_changepoints = (cp_probs > 0.5).sum()
        high_vol_days = (regime_labels == 1).sum()
        high_vol_pct = 100 * high_vol_days / len(regime_labels)
        print(f"    → Detected {n_changepoints} high-probability changepoints (p > 0.5)")
        print(f"    → High volatility regime: {high_vol_days} days ({high_vol_pct:.1f}%)")
    print()
    if Config.SAVE_RESULTS:
        import os

        print("[STEP 4] Saving results...")
        os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
        output_path = f"{Config.OUTPUT_DIR}bcpm_results.csv"
        aligned_df.to_csv(output_path, index=False)
        print(f"  → Saved results to {output_path}")
        summary_data = []
        for ticker in Config.TICKERS:
            regime_col = f"{ticker}_regime"
            cp_col = f"{ticker}_cp_prob"
            if regime_col in aligned_df.columns:
                summary_data.append(
                    {
                        "ticker": ticker,
                        "n_high_prob_changepoints": (aligned_df[cp_col] > 0.5).sum(),
                        "high_vol_days": (aligned_df[regime_col] == 1).sum(),
                        "high_vol_pct": 100 * (aligned_df[regime_col] == 1).sum() / len(aligned_df),
                        "avg_run_length": aligned_df[f"{ticker}_run_length"].mean(),
                        "avg_regime_vol": aligned_df[f"{ticker}_regime_vol"].mean(),
                    }
                )
        summary_df = pd.DataFrame(summary_data)
        summary_path = f"{Config.OUTPUT_DIR}bcpm_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"  → Saved summary to {summary_path}")
        print()
    print("[STEP 5] Creating visualizations...")
    visualizer = BCPMVisualizer()
    for ticker in Config.PLOT_ASSETS:
        if f"{ticker}_regime" in aligned_df.columns:
            print(f"  Plotting {ticker}...")
            save_path = None
            if Config.SAVE_RESULTS:
                save_path = f"{Config.OUTPUT_DIR}bcpm_{ticker}.png"
            visualizer.plot_changepoints_and_regimes(aligned_df, ticker, save_path=save_path)
    print("  Creating regime comparison chart...")
    save_path = None
    if Config.SAVE_RESULTS:
        save_path = f"{Config.OUTPUT_DIR}bcpm_regime_comparison.png"
    visualizer.plot_regime_comparison(aligned_df, Config.TICKERS, save_path=save_path)
    print()
    print("=" * 80)
    print("BCPM ANALYSIS COMPLETE")
    print("=" * 80)
    print()
    print("Key Outputs:")
    print("  1. bcpm_results.csv - Full dataset with all features and regime labels")
    print("  2. bcpm_summary.csv - Summary statistics per asset")
    print("  3. Individual asset plots showing changepoints and regimes")
    print("  4. Cross-asset regime comparison plot")
    print()
    print("Interpretation:")
    print("  - High changepoint probability (>0.5) indicates regime shift detected")
    print("  - Regime = 1: High volatility (red shading)")
    print("  - Regime = 0: Low volatility (no shading)")
    print("  - Run length: Time since last detected changepoint")
    print()
    print("Next Steps:")
    print("  - Use regime labels to adjust position sizing dynamically")
    print("  - Scale down exposure during high volatility regimes")
    print("  - Combine with other signals for regime-aware trading strategies")
    print()


if __name__ == "__main__":
    main()
